# 1. Can we use the downloaded data?

This notebook checks the **current 60-drug collection**. Think of it as checking a box of documents: are the files readable, are important fields missing, and did we keep the originals?

These checks help us prepare the files for searching. They do not prove that medical information is correct.
Run the cells from top to bottom. They use saved local files and make no API calls.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown

ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.utils.config import read_json

from src.ingestion.data_loader import load_snapshot

index_path = ROOT / read_json(ROOT / "results/search_latest.json")["run_path"]
index = read_json(index_path)
snapshot = load_snapshot(index["snapshot_id"])
coverage = index["coverage"]

## What did we collect?

| File type | What it means | Used for search? |
|---|---|---|
| Recent summary | A short preview from MORVS, a service that republishes FDA data | No |
| Name lookup | MORVS records found by searching a drug name; some text is shortened | No |
| Full FDA record | A complete JSON record downloaded from FDA; we index selected text sections | Yes |

One drug can have several labels from different products or manufacturers. A combination label can contain more than one drug, so the number of labels is different from the number of drug names.

In [2]:
source_names = {
    "recent_summary": "Recent summaries",
    "lookup": "Name-lookup records",
    "fda_full": "Full FDA records",
}
counts = pd.Series([record["kind"] for record in snapshot["records"]]).value_counts()
display(counts.rename(index=source_names).rename("Records").to_frame())
display(
    Markdown(
        f"**{coverage['covered_names']} drug names** are covered by **{coverage['full_fda_label_sets']} full FDA labels**. Each requested name has at least two matching labels."
    )
)

,Records
Name-lookup records,578
Recent summaries,200
Full FDA records,118


**60 drug names** are covered by **118 full FDA labels**. Each requested name has at least two matching labels.

## Did the files pass the basic checks?

We check that the saved files match their recorded digital fingerprints, the records have the expected structure, and full FDA text is available. Invalid records are set aside rather than used for searching.

In [3]:
from src.ingestion.integrity import verify_snapshot
from src.quality.validators import quality_report

integrity = verify_snapshot(snapshot)
quality = quality_report(snapshot)
summary = quality["summary"]
checks = {
    "Saved source files match their fingerprints": integrity["snapshot_integrity"] == "passed",
    "No records have structural errors": summary["schema_errors"] == 0,
    "No records needed to be set aside": summary["quarantined"] == 0,
    "Full FDA text is available": summary["acceptance"]["authoritative_corpus_available"],
}
display(pd.Series(checks).map({True: "Pass", False: "Needs attention"}).rename("Result").to_frame())
assert all(checks.values()), "Review the failed checks before continuing."

,Result
Saved source files match their fingerprints,Pass
No records have structural errors,Pass
No records needed to be set aside,Pass
Full FDA text is available,Pass


## What information is missing?

The table below checks the full FDA records. Some lack convenient name or product fields, even when the information appears in the label text. Missing fields can make filtering and searching harder.

NDC is a product identifier. RxCUI is an identifier used to connect drug names across systems.

In [4]:
field_names = {
    "generic_name": "Generic name",
    "brand_name": "Brand name",
    "manufacturer_name": "Manufacturer",
    "product_ndc": "Product identifier (NDC)",
    "rxcui": "Drug-name identifier (RxCUI)",
}
fields = pd.DataFrame(quality["fields"])
fields = fields[(fields["kind"] == "fda_full") & fields["field"].isin(field_names)].copy()
fields["Field"] = fields["field"].map(field_names)
fields["Records with value"] = fields["present"]
fields["Records missing value"] = fields["records"] - fields["present"]
display(fields[["Field", "Records with value", "Records missing value"]].set_index("Field"))

,Records with value,Records missing value
Field,,
Brand name,110,8
Generic name,110,8
Manufacturer,110,8
Product identifier (NDC),110,8
Drug-name identifier (RxCUI),109,9


## Why do we use the full FDA records?

We compared MORVS text with FDA text for labels with the same label date. Some MORVS sections stop before the FDA section ends. That is why the search index uses FDA text.

Warnings also flag unusual dates and possible text-encoding problems. A warning asks for inspection; it is different from a file failing to load.

Warning counts include all three source types. A text-length warning is only a clue, while the same-date text comparisons above directly show shortening.

In [5]:
comparisons = [row for row in snapshot["verification"] if row["same_effective_time"]]
shortened = sum(row["observed_truncation"] for row in comparisons)
display(
    Markdown(
        f"**{shortened} section comparisons** showed shortened MORVS text. This counts sections, not unique drugs."
    )
)
warnings = pd.DataFrame(quality["violations"])
warning_names = {
    "possible fixed-width truncation": "Text length suggests possible shortening",
    "future effective date": "Label date is in the future",
    "encoding or control-character anomaly": "Unusual or unreadable characters",
}
display(
    warnings[warnings["severity"] == "warning"]["issue"]
    .value_counts()
    .rename(index=warning_names)
    .rename("Occurrences")
    .to_frame()
)

**318 section comparisons** showed shortened MORVS text. This counts sections, not unique drugs.

,Occurrences
issue,
Text length suggests possible shortening,1244
Label date is in the future,6
Unusual or unreadable characters,2


## What did we learn?

The collection passed the basic file checks. Full FDA text avoids the observed shortening in MORVS records. Missing metadata and warnings still need attention when reviewing individual labels.

The 60 names were selected manually. This collection is a sample, not a list of every FDA drug. **Notebook 2 shows how its text becomes searchable passages.**